# The Shortcut Index

Reproduces the outputs of *Measuring Shortcut Contamination in Phishing Corpora: The Shortcut Index* (Shanta, Siam, Dipon and Mohaimen-Bin-Noor, AIUB).

Run the cells in order. **Runtime -> Run all** does the whole thing in about a minute; the optional TikZ cell at the end takes several minutes more because it installs LaTeX.

What you get:

1. Both result figures from the paper, regenerated from the reported numbers.
2. The index itself, run on corpora with known planted structure, so you can see what it responds to.
3. A scorer for your own CSV.
4. The methodology diagram (optional).
5. A zip of everything.

The index answers one question about a labelled dataset using nothing but that dataset: how much of its apparent separability is an artefact of how it was built?

```
SI(D) = (c1 + c2 + c3) / 3   in [0, 1]

c1  best single-feature threshold rule, chosen on training folds, scored held out
c2  |ARI| of k-means (k=2) against the label; labels score, never fit
c3  highest-confidence Apriori class rule, mined on training folds, re-scored held out
```

## Setup

In [ ]:
#@title Install dependencies and the Times-compatible font { display-mode: "form" }
!pip install -q mlxtend
!apt-get install -y -qq fonts-texgyre > /dev/null 2>&1

# The figures ask for TeX Gyre Termes so they match the paper's body font.
# Clearing the cache makes matplotlib notice the font we just installed.
import shutil, matplotlib
shutil.rmtree(matplotlib.get_cachedir(), ignore_errors=True)

import numpy, pandas, sklearn
print('numpy', numpy.__version__, '| pandas', pandas.__version__,
      '| scikit-learn', sklearn.__version__,
      '| matplotlib', matplotlib.__version__)

## The code

These three cells write the repository's files onto the Colab disk. They are the same files that produced the figures in the paper, embedded verbatim rather than re-typed, so anything you run here matches the submission.

In [ ]:
%%writefile shortcut_index.py
"""The Shortcut Index: how much of a corpus's separability is an artefact.

Reference implementation of the metric defined in Section III of "Measuring
Shortcut Contamination in Phishing Corpora: The Shortcut Index".

    SI(D) = (c1 + c2 + c3) / 3  in [0, 1]

Every component is estimated out of sample and rescaled against the
majority-class rate b = max(pi, 1 - pi), so a corpus no better than always
guessing the majority class scores 0:

    c1  single-feature separability   best one-attribute threshold rule,
                                      chosen on the training folds and scored
                                      on the held-out fold
    c2  unsupervised recoverability   |ARI| of k-means (k=2) against the label;
                                      labels score, never fit
    c3  rule determinism              highest-confidence Apriori class rule,
                                      mined on the training folds and
                                      re-scored on the held-out fold

Nothing here reads a second corpus. A high SI means reported accuracy on this
dataset will be dominated by construction artefacts, and that an ablation is
needed before any algorithm comparison means anything.

Command line
------------
    python shortcut_index.py data.csv --label is_phishing
    python shortcut_index.py data.csv --label y --null       # shuffled-label test
    python shortcut_index.py data.csv --label y --json out.json

Library
-------
    from shortcut_index import shortcut_index
    result = shortcut_index(X, y)      # DataFrame / array, binary labels
    print(result.si, result.c1, result.c2, result.c3)

Requires numpy, pandas and scikit-learn. mlxtend is used for c3 when it is
installed; without it c3 falls back to an equivalent exhaustive search over
itemsets up to `max_len`, which is what the paper's settings bound anyway.
"""

from __future__ import annotations

import argparse
import json
import sys
from dataclasses import dataclass, asdict
from typing import Iterable, Sequence

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

# Paper defaults (Section III-A and the Reproducibility note).
N_SPLITS = 5
SEED = 42
PERCENTILES = np.arange(1, 100)      # 1st to 99th percentile thresholds
KMEANS_RESTARTS = 10
MIN_SUPPORT = 0.05
MAX_LEN = 3
MIN_CONFIDENCE = 0.70
N_BINS = 3                           # equal-frequency tertiles


@dataclass
class SIResult:
    """One corpus scored. `raw_*` are before majority-baseline rescaling."""
    si: float
    c1: float
    c2: float
    c3: float
    raw_c1: float
    raw_c2: float
    raw_c3: float
    baseline: float
    n_rows: int
    n_features: int
    best_feature: str | None
    best_rule: str | None

    def __str__(self) -> str:
        return (
            f"SI = {self.si:.3f}\n"
            f"  c1 single-feature separability  {self.c1:.3f}"
            f"   (raw {self.raw_c1:.4f}, best: {self.best_feature})\n"
            f"  c2 unsupervised recoverability  {self.c2:.3f}"
            f"   (raw |ARI| {self.raw_c2:.4f})\n"
            f"  c3 rule determinism             {self.c3:.3f}"
            f"   (raw {self.raw_c3:.4f}, best: {self.best_rule})\n"
            f"  majority-class baseline         {self.baseline:.4f}"
            f"   over {self.n_rows:,} rows x {self.n_features} features"
        )


# ---------------------------------------------------------------------------
# helpers
# ---------------------------------------------------------------------------
def _rescale(score: float, baseline: float) -> float:
    """Map the majority-class rate to 0 and a perfect score to 1, clipped."""
    if baseline >= 1.0:
        return 0.0
    return float(np.clip((score - baseline) / (1.0 - baseline), 0.0, 1.0))


def _numeric_frame(X) -> pd.DataFrame:
    """Keep the numeric columns and fill gaps with the column median."""
    df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(np.asarray(X))
    df.columns = [str(c) for c in df.columns]
    num = df.select_dtypes(include=[np.number]).copy()
    if num.empty:
        raise ValueError("no numeric columns to score")
    return num.fillna(num.median(numeric_only=True)).replace(
        [np.inf, -np.inf], 0.0)


def _check_binary(y) -> np.ndarray:
    y = np.asarray(y).ravel()
    classes = np.unique(y)
    if len(classes) != 2:
        raise ValueError(f"expected two classes, found {len(classes)}")
    return (y == classes[1]).astype(int)


# ---------------------------------------------------------------------------
# c1: best single-feature threshold rule, cross-validated
# ---------------------------------------------------------------------------
def component_c1(X: pd.DataFrame, y: np.ndarray, n_splits: int = N_SPLITS,
                 seed: int = SEED) -> tuple[float, str | None]:
    """Mean held-out accuracy of the rule chosen on the training folds.

    The search is what makes this honest: the attribute, the threshold and the
    direction are all fixed on the training folds, then applied unchanged to
    the held-out fold. Selecting and scoring on the same rows would be
    optimistically biased, badly so on a small corpus.
    """
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    values = X.to_numpy(dtype=float)
    fold_scores, picks = [], []

    for train_idx, test_idx in cv.split(values, y):
        Xtr, ytr = values[train_idx], y[train_idx]
        Xte, yte = values[test_idx], y[test_idx]
        best = (-1.0, 0, 0.0, True)           # acc, column, threshold, above

        for j in range(Xtr.shape[1]):
            col = Xtr[:, j]
            thresholds = np.unique(np.percentile(col, PERCENTILES))
            if thresholds.size == 0:
                continue
            # vectorised over thresholds: (n_train, n_thresholds)
            above = col[:, None] > thresholds[None, :]
            acc_above = (above == ytr[:, None]).mean(axis=0)
            acc_below = 1.0 - acc_above
            k_above, k_below = int(acc_above.argmax()), int(acc_below.argmax())
            if acc_above[k_above] >= acc_below[k_below]:
                cand = (float(acc_above[k_above]), j,
                        float(thresholds[k_above]), True)
            else:
                cand = (float(acc_below[k_below]), j,
                        float(thresholds[k_below]), False)
            if cand[0] > best[0]:
                best = cand

        _, j, thr, above = best
        pred = (Xte[:, j] > thr) if above else (Xte[:, j] <= thr)
        fold_scores.append(float((pred == yte).mean()))
        picks.append((X.columns[j], thr, above))

    # report the attribute chosen most often across folds
    names = [p[0] for p in picks]
    best_name = max(set(names), key=names.count) if names else None
    return float(np.mean(fold_scores)), best_name


# ---------------------------------------------------------------------------
# c2: unsupervised recoverability
# ---------------------------------------------------------------------------
def component_c2(X: pd.DataFrame, y: np.ndarray, seed: int = SEED) -> float:
    """|ARI| between a 2-means partition and the label.

    The labels are used only to score the partition, never to fit it, so a high
    value means the two classes are already separated in the feature geometry
    before any supervision.
    """
    Z = StandardScaler().fit_transform(X.to_numpy(dtype=float))
    km = KMeans(n_clusters=2, n_init=KMEANS_RESTARTS, random_state=seed)
    return float(abs(adjusted_rand_score(y, km.fit_predict(Z))))


# ---------------------------------------------------------------------------
# c3: rule determinism
# ---------------------------------------------------------------------------
def _discretise(X: pd.DataFrame, n_bins: int = N_BINS) -> pd.DataFrame:
    """Equal-frequency bins; binary-valued columns are halved, as in the paper."""
    out = {}
    for col in X.columns:
        s = X[col]
        if s.nunique() <= 2:
            out[col] = s.astype(str)
            continue
        try:
            out[col] = pd.qcut(s, n_bins, duplicates="drop").astype(str)
        except ValueError:
            out[col] = s.astype(str)
    return pd.DataFrame(out, index=X.index)


def _onehot(bins: pd.DataFrame, y: np.ndarray) -> pd.DataFrame:
    items = pd.get_dummies(bins, prefix_sep="=").astype(bool)
    items["__class__=1"] = y.astype(bool)
    items["__class__=0"] = ~y.astype(bool)
    return items


def _best_class_rule(items: pd.DataFrame, min_support: float, max_len: int,
                     min_confidence: float):
    """Highest-confidence rule whose consequent is a class, mined on `items`.

    Returns (antecedent_columns, class_column, confidence) or None. Uses
    mlxtend's apriori when available and an equivalent bounded search when not.
    """
    class_cols = ["__class__=1", "__class__=0"]
    feature_cols = [c for c in items.columns if c not in class_cols]
    n = len(items)
    if n == 0:
        return None

    try:
        from mlxtend.frequent_patterns import apriori, association_rules
        freq = apriori(items, min_support=min_support, use_colnames=True,
                       max_len=max_len)
        if freq.empty:
            return None
        rules = association_rules(freq, metric="confidence",
                                  min_threshold=min_confidence)
        rules = rules[rules["consequents"].apply(
            lambda c: len(c) == 1 and next(iter(c)) in class_cols)]
        if rules.empty:
            return None
        top = rules.loc[rules["confidence"].idxmax()]
        return (list(top["antecedents"]), next(iter(top["consequents"])),
                float(top["confidence"]))
    except ImportError:
        pass

    # Fallback: the paper's support threshold and length cap make an explicit
    # search over antecedents of size 1..max_len-1 tractable.
    from itertools import combinations
    arrays = {c: items[c].to_numpy() for c in feature_cols}
    cls = {c: items[c].to_numpy() for c in class_cols}
    best = None
    for size in range(1, max_len):
        for combo in combinations(feature_cols, size):
            mask = arrays[combo[0]].copy()
            for c in combo[1:]:
                mask &= arrays[c]
            support = mask.mean()
            if support < min_support:
                continue
            covered = mask.sum()
            for c in class_cols:
                conf = float((mask & cls[c]).sum() / covered)
                if conf >= min_confidence and (best is None or conf > best[2]):
                    best = (list(combo), c, conf)
    return best


def component_c3(X: pd.DataFrame, y: np.ndarray, n_splits: int = N_SPLITS,
                 seed: int = SEED) -> tuple[float, str | None]:
    """Mean held-out confidence of the strongest class rule.

    Without the support threshold a rule covering a single record could reach
    confidence 1.000, so the threshold is doing real work here.
    """
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    bins = _discretise(X)
    baseline = max(float(np.mean(y)), 1.0 - float(np.mean(y)))
    scores, descriptions = [], []

    for train_idx, test_idx in cv.split(bins, y):
        tr = _onehot(bins.iloc[train_idx], y[train_idx])
        rule = _best_class_rule(tr, MIN_SUPPORT, MAX_LEN, MIN_CONFIDENCE)
        if rule is None:
            scores.append(baseline)       # no rule beats always-guess-majority
            continue
        antecedents, consequent, _ = rule

        te = _onehot(bins.iloc[test_idx], y[test_idx])
        missing = [c for c in antecedents + [consequent] if c not in te.columns]
        if missing:
            scores.append(baseline)
            continue
        mask = te[antecedents[0]].to_numpy().copy()
        for c in antecedents[1:]:
            mask &= te[c].to_numpy()
        covered = int(mask.sum())
        if covered == 0:
            scores.append(baseline)
            continue
        held_out = float((mask & te[consequent].to_numpy()).sum() / covered)
        scores.append(held_out)
        descriptions.append(
            f"{' AND '.join(antecedents)} -> {consequent}")

    return float(np.mean(scores)), (descriptions[0] if descriptions else None)


# ---------------------------------------------------------------------------
# the index
# ---------------------------------------------------------------------------
def shortcut_index(X, y, n_splits: int = N_SPLITS, seed: int = SEED,
                   shuffle_labels: bool = False) -> SIResult:
    """Score one corpus. Set `shuffle_labels` to run the null test.

    Under permutation the feature distributions are untouched and only the
    label assignment changes, so a well-behaved index collapses towards zero.
    """
    Xn = _numeric_frame(X)
    yb = _check_binary(y)
    if len(Xn) != len(yb):
        raise ValueError("X and y have different lengths")
    if shuffle_labels:
        yb = np.random.default_rng(seed).permutation(yb)

    baseline = max(float(np.mean(yb)), 1.0 - float(np.mean(yb)))
    raw_c1, best_feature = component_c1(Xn, yb, n_splits, seed)
    raw_c2 = component_c2(Xn, yb, seed)
    raw_c3, best_rule = component_c3(Xn, yb, n_splits, seed)

    c1 = _rescale(raw_c1, baseline)
    c2 = float(np.clip(raw_c2, 0.0, 1.0))     # ARI is already baseline-free
    c3 = _rescale(raw_c3, baseline)

    return SIResult(
        si=(c1 + c2 + c3) / 3.0, c1=c1, c2=c2, c3=c3,
        raw_c1=raw_c1, raw_c2=raw_c2, raw_c3=raw_c3, baseline=baseline,
        n_rows=len(Xn), n_features=Xn.shape[1],
        best_feature=best_feature, best_rule=best_rule,
    )


def generalisation_gap(in_domain: float, external: Iterable[float]) -> float:
    """In-domain accuracy minus mean external accuracy (Section III-B)."""
    ext = list(external)
    if not ext:
        raise ValueError("need at least one external accuracy")
    return float(in_domain) - float(np.mean(ext))


# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------
def main(argv: Sequence[str] | None = None) -> int:
    ap = argparse.ArgumentParser(description=__doc__.split("\n")[0])
    ap.add_argument("csv", help="labelled corpus; one row per record")
    ap.add_argument("--label", required=True, help="name of the label column")
    ap.add_argument("--drop", nargs="*", default=[],
                    help="columns to exclude (ids, free text, raw URLs)")
    ap.add_argument("--folds", type=int, default=N_SPLITS)
    ap.add_argument("--seed", type=int, default=SEED)
    ap.add_argument("--null", action="store_true",
                    help="also run the shuffled-label null test")
    ap.add_argument("--json", metavar="PATH", help="write the result as JSON")
    args = ap.parse_args(argv)

    df = pd.read_csv(args.csv)
    if args.label not in df.columns:
        print(f"label column {args.label!r} not in {args.csv}", file=sys.stderr)
        return 2
    y = df[args.label]
    X = df.drop(columns=[args.label] + [c for c in args.drop if c in df.columns])

    result = shortcut_index(X, y, args.folds, args.seed)
    print(f"{args.csv}\n{result}")

    payload = {"corpus": args.csv, **asdict(result)}
    if args.null:
        null = shortcut_index(X, y, args.folds, args.seed, shuffle_labels=True)
        print(f"\nshuffled labels: SI = {null.si:.4f} "
              f"(c1 {null.c1:.3f}, c2 {null.c2:.3f}, c3 {null.c3:.3f})")
        drop = result.si - null.si
        print(f"the index drops by {drop:.4f}; a value that does not drop is "
              f"responding to the feature distributions, not the labels")
        payload["null"] = asdict(null)

    if result.si >= 0.5:
        print("\nSI at or above 0.5: ablate the top single-feature attributes "
              "before reporting accuracy or comparing algorithms.")

    if args.json:
        with open(args.json, "w") as fh:
            json.dump(payload, fh, indent=2)
        print(f"\nwrote {args.json}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile paper_results.py
"""Reported results for 'Measuring Shortcut Contamination in Phishing Corpora'.

Single source of truth for every number that appears in a table or a figure of
the paper. `make_figures.py` reads from here, so a corrected measurement only
has to be edited once.

Values produced by `shortcut_index.py` on the corpora listed in Section IV.
"""

# ---------------------------------------------------------------------------
# Table I: Shortcut Index components and observed generalisation gap.
# gap = in-domain accuracy - mean external accuracy (decision tree, 15-feature
# portable subset).
# ---------------------------------------------------------------------------
CORPORA = ["Enron", "SpamAssassin", "Ling", "CEAS-08", "SMS-Spam"]

SI_TABLE = {
    #                c1      c2      c3      SI     gap
    "Enron":        (0.319, 0.000, 0.729, 0.349, 0.292),
    "SpamAssassin": (0.311, 0.015, 1.000, 0.442, 0.277),
    "Ling":         (0.500, 0.109, 0.895, 0.501, 0.227),
    "CEAS-08":      (0.667, 0.008, 0.965, 0.547, 0.414),
    "SMS-Spam":     (0.763, 0.489, 0.972, 0.741, 0.383),
}

# Rank correlation between SI and the measured gap over the five corpora above.
SI_GAP_RHO = 0.500
SI_GAP_P = 0.391
SI_GAP_BOOT_CI = (-1.000, 0.875)
SI_GAP_PERM_P = 0.446

# ---------------------------------------------------------------------------
# Table II: shuffled-label null test. Labels permuted, feature distributions
# untouched, SI recomputed.
# ---------------------------------------------------------------------------
NULL_TEST = {
    #                SI_true  SI_shuffled  c1_shuffled  c3_shuffled
    "Enron":        (0.349, 0.0001, 0.000, 0.000),
    "SpamAssassin": (0.442, 0.0005, 0.000, 0.000),
    "Ling":         (0.501, 0.0505, 0.000, 0.142),
    "CEAS-08":      (0.547, 0.0003, 0.000, 0.000),
    "SMS-Spam":     (0.741, 0.0076, 0.000, 0.000),
}

# ---------------------------------------------------------------------------
# Table III: balanced accuracy by classifier family, trained on CEAS-08 and
# applied to the four other corpora. Same 15-feature portable subset throughout.
# ---------------------------------------------------------------------------
CLASSIFIER_FAMILIES = [
    # name,                in-domain, external, gap
    ("Naive Bayes",          0.587, 0.426, 0.161),
    ("Logistic regression",  0.921, 0.551, 0.370),
    ("Gradient boosting",    0.967, 0.591, 0.376),
    ("Random forest",        0.986, 0.587, 0.399),
    ("Decision tree",        0.975, 0.561, 0.414),
    ("KNN ($k=3$)",          0.959, 0.541, 0.418),
]

# ---------------------------------------------------------------------------
# Figure: decision tree / KNN / naive Bayes trained on CEAS-08, applied
# unchanged to five external email corpora.
# ---------------------------------------------------------------------------
TRANSFER_TARGETS = [
    "CEAS-08\n(in-domain)", "SpamAssassin", "Enron", "Ling",
    "Nazario", "Nigerian\nFraud",
]
TRANSFER = {
    "KNN":           [0.9383, 0.663, 0.528, 0.541, 0.403, 0.434],
    "Naive Bayes":   [0.6657, 0.246, 0.438, 0.367, 0.644, 0.470],
    "Decision Tree": [0.9786, 0.702, 0.610, 0.768, 0.302, 0.286],
}

# ---------------------------------------------------------------------------
# Table IV: classification performance on the synthetic corpus (Syn.), the
# same corpus after the five-regime shortcut ablation (Abl.), and CEAS-08.
# ---------------------------------------------------------------------------
PERFORMANCE = [
    # block, model,           acc,    prec,   rec,    f1,     auc,    cv
    ("Syn.", "KNN",           0.9924, 1.0000, 0.9813, 0.9906, 0.9911, 0.9913),
    ("Syn.", "Naive Bayes",   0.9926, 1.0000, 0.9818, 0.9908, 0.9893, 0.9917),
    ("Syn.", "Decision Tree", 0.9935, 0.9952, 0.9887, 0.9920, 0.9981, 0.9919),
    ("Abl.", "KNN",           0.8049, 0.7889, 0.7117, 0.7483, 0.8873, 0.7995),
    ("Abl.", "Naive Bayes",   0.8224, 0.7580, 0.8287, 0.7917, 0.8964, 0.8209),
    ("Abl.", "Decision Tree", 0.8531, 0.8303, 0.8038, 0.8168, 0.9275, 0.8537),
    ("CEAS", "KNN",           0.9383, 0.9393, 0.9509, 0.9450, 0.9719, 0.9292),
    ("CEAS", "Naive Bayes",   0.6657, 0.6368, 0.9325, 0.7568, 0.7891, 0.6585),
    ("CEAS", "Decision Tree", 0.9786, 0.9787, 0.9831, 0.9809, 0.9788, 0.9767),
]

# ---------------------------------------------------------------------------
# Table V: SI components for PhiUSIIL (2024), scored out of sample.
# ---------------------------------------------------------------------------
PHIUSIIL = {
    "c1 best single-feature rule": (0.9967, 0.992),
    "c2 unsupervised k-means ARI": (0.9453, 0.945),
    "c3 max class-rule confidence": (1.0000, 1.000),
}
PHIUSIIL_SI = 0.979

# ---------------------------------------------------------------------------
# Table VI: release year against SI. `ref` is the citation key used in the
# paper for corpora that are not one of the five primary sets.
# ---------------------------------------------------------------------------
RECENCY = [
    # corpus,            year, best single feature, SI
    ("Ling",             2000, 0.920, 0.501),
    ("SpamAssassin",     2002, 0.876, 0.442),
    ("Enron",            2004, 0.680, 0.349),
    ("CEAS-08",          2008, 0.853, 0.547),
    ("PhiUSIIL",         2024, 0.997, 0.979),
    ("URL corpus '25",   2025, 1.000, 0.998),
    ("Synthetic email",  2026, 0.992, 0.654),
    ("URL corpus '26",   2026, 0.947, 0.789),
]

# ---------------------------------------------------------------------------
# Table VII: effect sizes. Every modelling decision we varied against every
# dataset property we varied, on one accuracy scale.
# ---------------------------------------------------------------------------
EFFECT_SIZES = [
    # kind, label, setting, delta accuracy
    ("knob", "Splitting criterion", "entropy vs. Gini", 0.0027),
    ("knob", "Estimation protocol", "hold-out vs. 10-fold CV", 0.0026),
    ("knob", "Training-set size", "25\\% vs. 100\\%", 0.0048),
    ("knob", "Hold-out seed", "20 seeds, identical data", 0.0068),
    ("data", "Shortcut attributes", "synthetic, regime A vs. E", 0.1404),
    ("data", "Corpus transfer", "CEAS-08 in-domain vs. external", 0.3270),
    ("data", "Modality transfer", "CEAS-08 vs. smishing recall", 0.5070),
]

# ---------------------------------------------------------------------------
# Section V-I: recall on 22,003 real English smishing messages for a detector
# trained on CEAS-08.
# ---------------------------------------------------------------------------
SMISHING_RECALL = {"Decision Tree": 0.470, "KNN": 0.234, "Naive Bayes": 0.187}
SMISHING_BY_LURE = {"Wrong number": 0.603, "Government impersonation": 0.367}

In [ ]:
%%writefile make_figures.py
"""Regenerate every figure in the paper from the numbers in `paper_results.py`.

    python make_figures.py [--outdir ../paper/figs] [--png]

Writes vector PDF (what the LaTeX build includes) and, with --png, 300 dpi
raster copies for the Word version. Two composite figures come out:

  fig_index   the index itself and its validation (components, SI vs. gap,
              shuffled-label null, release year)
  fig_gap     where the accuracy goes (cross-corpus transfer, classifier
              families, effect sizes)

Both are laid out for a double-column IEEE float. The panel grouping is what
lets the paper carry six figures' worth of evidence inside a six-page limit.

Chart conventions
-----------------
Colours are the first three slots of a categorical palette validated for
colour-vision deficiency at all pairs (worst CVD dE 9.2, normal-vision dE 24.0).
Because conference papers are routinely printed in greyscale, hue is never the
only channel: every fill also carries a hatch at 45 degrees or its 135 degree
mirror, and values are labelled directly wherever the panel has room. One value
axis per panel, no dual scales. Grid and axis lines are recessive; text stays in
neutral ink, so a coloured mark beside a label carries the identity rather than
the label itself.
"""

import argparse
import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.patheffects as pe
from matplotlib.patches import Patch

import paper_results as R

# --- palette -----------------------------------------------------------------
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, AXIS = "#e1e0d9", "#c3c2b7"
HATCH = ["", "///", "\\\\\\"]      # none, 45 deg, its 135 deg mirror

COL_W, FULL_W = 3.45, 7.10          # IEEE single- and double-column widths (in)


def style():
    plt.rcParams.update({
        "font.family": "serif",
        "font.serif": ["TeX Gyre Termes", "Times New Roman", "DejaVu Serif"],
        "mathtext.fontset": "stix",
        "font.size": 6.8,
        "axes.titlesize": 7.2,
        "axes.labelsize": 6.8,
        "xtick.labelsize": 6.3,
        "ytick.labelsize": 6.3,
        "legend.fontsize": 6.1,
        "axes.edgecolor": AXIS,
        "axes.linewidth": 0.6,
        "axes.labelcolor": INK2,
        "text.color": INK,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "xtick.major.width": 0.6,
        "ytick.major.width": 0.6,
        "grid.color": GRID,
        "grid.linewidth": 0.6,
        "hatch.linewidth": 0.4,
        "figure.dpi": 300,
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.015,
        "pdf.fonttype": 42,
    })


def tidy(ax, xgrid=False):
    """Recessive chrome: drop the top and right rules, keep one faint grid."""
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.spines["left"].set_color(AXIS)
    ax.spines["bottom"].set_color(AXIS)
    ax.set_axisbelow(True)
    ax.grid(axis="x" if xgrid else "y", linestyle="-", alpha=0.9)
    ax.tick_params(length=2.0, pad=1.6)


def title(ax, text):
    ax.set_title(text, color=INK2, pad=3.5, fontsize=7.2)


def save(fig, outdir, name, png):
    fig.savefig(os.path.join(outdir, name + ".pdf"))
    if png:
        fig.savefig(os.path.join(outdir, name + ".png"), dpi=300)
    plt.close(fig)
    print("  wrote", name)


# ===========================================================================
# Panels
# ===========================================================================
def panel_components(ax):
    """(a) The three diagnostics and the composite, per corpus."""
    order = sorted(R.CORPORA, key=lambda c: R.SI_TABLE[c][3])
    series = [
        ([R.SI_TABLE[c][0] for c in order], BLUE, r"$c_1$"),
        ([R.SI_TABLE[c][1] for c in order], ORANGE, r"$c_2$"),
        ([R.SI_TABLE[c][2] for c in order], AQUA, r"$c_3$"),
    ]
    si = [R.SI_TABLE[c][3] for c in order]
    x = np.arange(len(order))
    w = 0.26

    for k, (vals, colour, lab) in enumerate(series):
        ax.bar(x + (k - 1) * w, vals, w * 0.92, color=colour, hatch=HATCH[k],
               edgecolor="white", linewidth=0.5, label=lab, zorder=3)
    ax.plot(x, si, linestyle="none", marker="D", markersize=3.6,
            markerfacecolor="white", markeredgecolor=INK, markeredgewidth=0.9,
            zorder=5, label="SI")
    for xi, s in zip(x, si):
        ax.annotate(f"{s:.2f}", (xi, s), textcoords="offset points",
                    xytext=(0, 5), ha="center", fontsize=5.8, color=INK)

    ax.set_xticks(x)
    ax.set_xticklabels([c.replace("SpamAssassin", "SpamA.").replace("CEAS-08", "CEAS")
                        .replace("SMS-Spam", "SMS") for c in order],
                       fontsize=5.9)
    ax.set_ylim(0, 1.32)
    ax.set_yticks([0, 0.5, 1.0])
    ax.set_ylabel("Normalised score")
    tidy(ax)
    ax.legend(ncol=4, frameon=False, loc="upper left", handlelength=1.0,
              columnspacing=0.8, handletextpad=0.35, labelcolor=INK2,
              borderpad=0.05, fontsize=6.1)
    title(ax, "(a) Components of SI")


def panel_si_gap(ax):
    """(b) SI against the measured generalisation gap."""
    si = np.array([R.SI_TABLE[c][3] for c in R.CORPORA])
    gap = np.array([R.SI_TABLE[c][4] for c in R.CORPORA])
    ax.scatter(si, gap, s=26, color=BLUE, edgecolor="white", linewidth=0.6,
               zorder=4)
    offsets = {"Enron": (5, -2), "SpamAssassin": (5, -2), "Ling": (5, -2),
               "CEAS-08": (-5, -2), "SMS-Spam": (-5, -2)}
    for c, xs, ys in zip(R.CORPORA, si, gap):
        dx, dy = offsets[c]
        ax.annotate(c.replace("SpamAssassin", "SpamAss."), (xs, ys),
                    textcoords="offset points", xytext=(dx, dy), fontsize=5.8,
                    color=INK2, ha="right" if dx < 0 else "left")
    ax.set_xlim(0.22, 0.92)
    ax.set_ylim(0.17, 0.47)
    ax.set_xticks([0.3, 0.5, 0.7, 0.9])
    ax.set_xlabel("Shortcut Index")
    ax.set_ylabel("Measured gap")
    tidy(ax)
    ax.annotate(f"$\\rho$ = {R.SI_GAP_RHO:.2f}\n$p$ = {R.SI_GAP_P:.2f}\n$n$ = 5",
                (0.97, 0.04), xycoords="axes fraction", va="bottom",
                ha="right", fontsize=6.0, color=INK2)
    title(ax, "(b) SI vs. observed gap")


def panel_null(ax):
    """(c) True against shuffled-label SI, on a log axis."""
    order = sorted(R.CORPORA, key=lambda c: R.NULL_TEST[c][0])
    y = np.arange(len(order))
    true = [R.NULL_TEST[c][0] for c in order]
    shuf = [max(R.NULL_TEST[c][1], 1e-4) for c in order]
    for yi, t, s in zip(y, true, shuf):
        ax.plot([s, t], [yi, yi], color=AXIS, linewidth=1.0, zorder=2)
    ax.scatter(shuf, y, s=20, color=ORANGE, marker="s", edgecolor="white",
               linewidth=0.5, zorder=4)
    ax.scatter(true, y, s=22, color=BLUE, edgecolor="white", linewidth=0.5,
               zorder=4)
    top = len(order) - 1
    ax.annotate("shuffled", (shuf[top], top), textcoords="offset points",
                xytext=(0, 6), ha="center", fontsize=5.8, color=ORANGE)
    ax.annotate("true", (true[top], top), textcoords="offset points",
                xytext=(0, 6), ha="center", fontsize=5.8, color=BLUE)
    ax.set_xscale("log")
    ax.set_xlim(4e-5, 4.0)
    ax.set_xticks([1e-4, 1e-2, 1e0])
    ax.set_yticks(y)
    ax.set_yticklabels([c.replace("SpamAssassin", "SpamAss.") for c in order],
                       fontsize=6.0)
    ax.set_ylim(-0.55, len(order) + 0.15)
    ax.set_xlabel("SI (log scale)")
    tidy(ax, xgrid=True)
    title(ax, "(c) Shuffled-label null")


def panel_recency(ax):
    """(d) Release year against SI for all eight scored corpora."""
    email = [r for r in R.RECENCY if r[1] < 2020]
    recent = [r for r in R.RECENCY if r[1] >= 2020]
    ax.scatter([r[1] for r in email], [r[3] for r in email], s=24, color=BLUE,
               edgecolor="white", linewidth=0.5, zorder=4,
               label="2000–2008")
    ax.scatter([r[1] for r in recent], [r[3] for r in recent], s=24,
               color=ORANGE, marker="^", edgecolor="white", linewidth=0.5,
               zorder=4, label="2024–2026")
    ax.set_xlim(1996, 2031)
    ax.set_ylim(0, 1.22)
    ax.set_xticks([2000, 2010, 2020, 2030])
    ax.set_yticks([0, 0.5, 1.0])
    ax.set_xlabel("Release year")
    ax.set_ylabel("Shortcut Index")
    tidy(ax)
    ax.legend(frameon=False, loc="upper left", handletextpad=0.2,
              labelcolor=INK2, borderpad=0.05, labelspacing=0.15, fontsize=6.0)
    title(ax, "(d) Newer is not cleaner")


def panel_transfer(ax):
    """(a) CEAS-08 detectors applied unchanged to five external corpora."""
    labels = ["CEAS-08\nin-domain", "Spam\nAssassin", "Enron", "Ling",
              "Nazario", "Nigerian\nFraud"]
    x = np.arange(len(labels))
    w = 0.27
    for k, (name, colour) in enumerate([("KNN", BLUE),
                                        ("Naive Bayes", ORANGE),
                                        ("Decision Tree", AQUA)]):
        ax.bar(x + (k - 1) * w, R.TRANSFER[name], w * 0.9, color=colour,
               hatch=HATCH[k], edgecolor="white", linewidth=0.5, label=name,
               zorder=3)
    ax.axvline(0.5, color=INK, linestyle=(0, (3, 2)), linewidth=0.8, zorder=4)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=5.7)
    ax.set_ylim(0, 1.30)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.set_ylabel("Accuracy")
    tidy(ax)
    ax.legend(ncol=3, frameon=False, loc="upper center",
              bbox_to_anchor=(0.55, 1.04), handlelength=1.1,
              columnspacing=1.0, handletextpad=0.35, labelcolor=INK2,
              borderpad=0.05, fontsize=6.1)
    title(ax, "(a) Trained on CEAS-08, applied unchanged")


def panel_families(ax):
    """(b) Six classifier families, in-domain against external."""
    fam = sorted(R.CLASSIFIER_FAMILIES, key=lambda r: r[3])
    names = [f[0].replace("$k=3$", "k=3").replace("Logistic regression",
                                                  "Logistic reg.") for f in fam]
    y = np.arange(len(fam))
    h = 0.36
    ax.barh(y + h / 2, [f[1] for f in fam], h * 0.9, color=BLUE,
            edgecolor="white", linewidth=0.45, zorder=3)
    ax.barh(y - h / 2, [f[2] for f in fam], h * 0.9, color=ORANGE,
            hatch=HATCH[1], edgecolor="white", linewidth=0.45, zorder=3)
    top = len(fam) - 1
    halo = [pe.withStroke(linewidth=1.4, foreground=INK, alpha=0.55)]
    ax.annotate("in-domain", (0.03, top + h / 2), va="center", fontsize=5.6,
                color="white", zorder=6, path_effects=halo)
    ax.annotate("external", (0.03, top - h / 2), va="center", fontsize=5.6,
                color="white", zorder=6, path_effects=halo)
    for yi, f in zip(y, fam):
        ax.annotate(f"−{f[3]:.2f}", (max(f[1], f[2]), yi),
                    textcoords="offset points", xytext=(3, -2), fontsize=5.8,
                    color=INK2)
    ax.set_yticks(y)
    ax.set_yticklabels(names, fontsize=6.0)
    ax.set_xlim(0, 1.34)
    ax.set_xticks([0, 0.5, 1.0])
    ax.set_xlabel("Balanced accuracy")
    tidy(ax, xgrid=True)
    title(ax, "(b) Every family collapses")


def panel_effects(ax):
    """(c) Modelling knobs and dataset properties on one accuracy scale."""
    eff = sorted(R.EFFECT_SIZES, key=lambda r: r[3])
    y = np.arange(len(eff))
    bars = ax.barh(y, [r[3] for r in eff], 0.6,
                   color=[BLUE if r[0] == "knob" else ORANGE for r in eff],
                   edgecolor="white", linewidth=0.45, zorder=3)
    for b, r in zip(bars, eff):
        b.set_hatch("" if r[0] == "knob" else HATCH[1])
    for yi, r in zip(y, eff):
        ax.annotate(f"{r[3]:.4f}".rstrip("0"), (r[3], yi),
                    textcoords="offset points", xytext=(3, -2), fontsize=5.8,
                    color=INK2)
    ax.set_xscale("log")
    ax.set_xlim(1.2e-3, 5.0)
    ax.set_xticks([1e-2, 1e-1, 1e0])
    ax.set_yticks(y)
    ax.set_yticklabels([r[1].replace("Estimation protocol", "Est. protocol")
                        for r in eff], fontsize=6.0)
    ax.set_xlabel(r"$\Delta$ accuracy (log scale)")
    tidy(ax, xgrid=True)
    ax.legend(handles=[Patch(facecolor=BLUE, edgecolor="white",
                             label="Modelling knob"),
                       Patch(facecolor=ORANGE, hatch=HATCH[1],
                             edgecolor="white", label="Dataset property")],
              frameon=False, loc="lower right", bbox_to_anchor=(1.02, -0.03),
              handlelength=1.0, handletextpad=0.3, labelcolor=INK2,
              borderpad=0.05, labelspacing=0.15, fontsize=6.0)
    title(ax, "(c) Knobs vs. dataset properties")


# ===========================================================================
# Composite figures
# ===========================================================================
def fig_index(outdir, png):
    fig, axes = plt.subplots(
        1, 4, figsize=(FULL_W, 1.46),
        gridspec_kw={"width_ratios": [1.30, 0.92, 0.94, 0.84], "wspace": 0.44})
    panel_components(axes[0])
    panel_si_gap(axes[1])
    panel_null(axes[2])
    panel_recency(axes[3])
    save(fig, outdir, "fig_index", png)


def fig_gap(outdir, png):
    fig, axes = plt.subplots(
        1, 3, figsize=(FULL_W, 1.46),
        gridspec_kw={"width_ratios": [1.85, 1.02, 1.05], "wspace": 0.44})
    panel_transfer(axes[0])
    panel_families(axes[1])
    panel_effects(axes[2])
    save(fig, outdir, "fig_gap", png)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--outdir", default=os.path.join(
        os.path.dirname(os.path.abspath(__file__)), "..", "paper", "figs"))
    ap.add_argument("--png", action="store_true",
                    help="also write 300 dpi PNG copies for the Word version")
    args = ap.parse_args()

    outdir = os.path.abspath(args.outdir)
    os.makedirs(outdir, exist_ok=True)
    style()
    print("figures ->", outdir)
    fig_index(outdir, args.png)
    fig_gap(outdir, args.png)


if __name__ == "__main__":
    main()

## 1. Regenerate the paper's figures

`paper_results.py` holds every number that appears in a table or figure. Change a value there and every figure that uses it updates, which is how the paper and the plots are kept in step.

In [ ]:
!python make_figures.py --outdir figs --png

from IPython.display import Image, display
display(Image('figs/fig_index.png', width=1000))
display(Image('figs/fig_gap.png', width=1000))

**Fig. 2 (top)** the index and its validation: (a) the three diagnostics and the composite per corpus, (b) SI against the measured generalisation gap, (c) the shuffled-label null on a log axis, (d) release year against SI.

**Fig. 3 (bottom)** where the accuracy goes: (a) CEAS-08 detectors applied unchanged to five external corpora, (b) six classifier families in-domain against external, (c) every quantity the paper varied, on one accuracy scale.

## 2. What the index actually responds to

Three corpora with known structure. The first has one attribute that is the label under another name, which is the failure mode the paper is about. The second has no signal at all. The third has a real but modest association, which is not contamination and should score low.

The null column permutes the labels and leaves every feature distribution untouched. An index that measures label-bearing structure has to collapse; one that does not is reading quirks of the features.

In [ ]:
import numpy as np, pandas as pd
from shortcut_index import shortcut_index

rng = np.random.default_rng(0)
n = 1500
y = rng.integers(0, 2, n)

corpora = {
    'planted shortcut': pd.DataFrame({
        # exactly the urgency_score pattern of Section V-E: a gap in the range,
        # every record above it phishing
        'urgency_score': np.where(y == 1, rng.integers(7, 11, n),
                                  rng.integers(1, 6, n)),
        'body_len': rng.normal(50, 10, n),
        'n_links': rng.poisson(3, n),
        'entropy': rng.normal(4.2, 0.3, n),
    }),
    'pure noise': pd.DataFrame(
        {f'f{i}': rng.normal(size=n) for i in range(4)}),
    'weak genuine signal': pd.DataFrame({
        'f0': rng.normal(size=n) + 0.45 * y,
        'f1': rng.normal(size=n),
        'f2': rng.normal(size=n),
    }),
}

rows = []
for name, X in corpora.items():
    r = shortcut_index(X, y)
    null = shortcut_index(X, y, shuffle_labels=True)
    rows.append({'corpus': name, 'c1': r.c1, 'c2': r.c2, 'c3': r.c3,
                 'SI': r.si, 'SI (labels shuffled)': null.si,
                 'best feature': r.best_feature})

pd.set_option('display.width', 140)
pd.DataFrame(rows).round(4)

## 3. Score your own corpus

Upload a CSV with one row per record and a binary label column. Everything numeric is scored; drop identifiers, free text and raw URLs, because an id column will separate the classes perfectly and tell you nothing.

Reading the number: SI is a screening heuristic, not a calibrated predictor of how much accuracy transfers. Above roughly 0.5 in the paper's sample, ablate the top single-feature attributes before comparing algorithms. A low value rules out this internal failure mode and does not certify the corpus, so report SI alongside at least one external evaluation.

In [ ]:
import io, os
import pandas as pd
from shortcut_index import shortcut_index

LABEL = 'label'      #@param {type:"string"}
DROP  = ''           #@param {type:"string"}

try:
    from google.colab import files
    uploaded = files.upload()
    name = next(iter(uploaded))
    df = pd.read_csv(io.BytesIO(uploaded[name]))
except ImportError:
    name = 'your_corpus.csv'
    df = pd.read_csv(name) if os.path.exists(name) else None

if df is None:
    print('Not running in Colab. Put a CSV beside this notebook '
          'or call shortcut_index(X, y) directly.')
elif LABEL not in df.columns:
    print(f'No column named {LABEL!r}. Columns are:',
          list(df.columns))
else:
    drop = [c.strip() for c in DROP.split(',') if c.strip() in df.columns]
    X = df.drop(columns=[LABEL] + drop)
    result = shortcut_index(X, df[LABEL])
    null = shortcut_index(X, df[LABEL], shuffle_labels=True)
    print(f'{name}\n{result}')
    print(f'\nshuffled labels: SI = {null.si:.4f} '
          f'(drops by {result.si - null.si:.4f})')
    if result.si >= 0.5:
        print('\nSI at or above 0.5: ablate the top single-feature attributes'
              ' before reporting accuracy or comparing algorithms.')

### Sanity checks

Eight behaviour tests: a corpus whose label is renamed as an attribute scores above 0.9, pure noise below 0.05, a weak genuine association below 0.25, and all three collapse under label permutation.

In [ ]:
%%writefile test_quick.py
import numpy as np, pandas as pd
from shortcut_index import shortcut_index

rng = np.random.default_rng(0); n = 1500
y = rng.integers(0, 2, n)
planted = pd.DataFrame({
    'urgency_score': np.where(y == 1, rng.integers(7, 11, n),
                              rng.integers(1, 6, n)),
    'a': rng.normal(size=n), 'b': rng.normal(size=n),
    'c': rng.normal(50, 10, n)})
noise = pd.DataFrame({f'f{i}': rng.normal(size=n) for i in range(4)})
weak = pd.DataFrame({'f0': rng.normal(size=n) + 0.45 * y,
                     'f1': rng.normal(size=n), 'f2': rng.normal(size=n)})

checks = [
    ('planted shortcut scores high', shortcut_index(planted, y).si > 0.9),
    ('pure noise scores zero',      shortcut_index(noise, y).si < 0.05),
    ('weak signal scores low',      shortcut_index(weak, y).si < 0.25),
]
for X, label in ((planted, 'planted'), (noise, 'noise'), (weak, 'weak')):
    checks.append((f'null test collapses ({label})',
                   shortcut_index(X, y, shuffle_labels=True).si < 0.06))
r = shortcut_index(planted, y)
checks.append(('components bounded',
               all(0 <= v <= 1 for v in (r.si, r.c1, r.c2, r.c3))))
checks.append(('SI is the mean of its components',
               abs(r.si - (r.c1 + r.c2 + r.c3) / 3) < 1e-9))

for name, ok in checks:
    print(('  ok   ' if ok else '  FAIL '), name)
print(f"\n{sum(ok for _, ok in checks)}/{len(checks)} passed")

In [ ]:
!python test_quick.py

## 4. The methodology diagram (optional)

Fig. 1 is TikZ rather than matplotlib, so this cell installs LaTeX. It takes a few minutes and roughly 200 MB. Skip it unless you want to edit the diagram; the compiled PDF is already in the LaTeX package.

In [ ]:
#@title Install LaTeX and build the diagram { display-mode: "form" }
BUILD_DIAGRAM = False  #@param {type:"boolean"}

if BUILD_DIAGRAM:
    !apt-get install -y -qq texlive-latex-base texlive-pictures \
        texlive-latex-extra texlive-fonts-recommended > /dev/null 2>&1
    import urllib.request, os
    if not os.path.exists('methodology.tex'):
        print('Upload methodology.tex from the LaTeX package, '
              'then re-run this cell.')
        try:
            from google.colab import files
            files.upload()
        except ImportError:
            pass
    if os.path.exists('methodology.tex'):
        !pdflatex -interaction=nonstopmode methodology.tex > /dev/null 2>&1
        !pdftoppm -png -r 200 methodology.pdf figs/methodology
        !mv figs/methodology-1.png figs/methodology.png
        from IPython.display import Image, display
        display(Image('figs/methodology.png', width=560))
else:
    print('Skipped. Set BUILD_DIAGRAM = True to compile Fig. 1.')

## 5. Download the outputs

In [ ]:
import glob, os

!zip -qr shortcut_index_outputs.zip figs *.py
print('in the zip:')
for f in sorted(glob.glob('figs/*') + glob.glob('*.py')):
    print(f'  {f}  ({os.path.getsize(f) / 1024:.0f} KB)')

try:
    from google.colab import files
    files.download('shortcut_index_outputs.zip')
except ImportError:
    print('\nNot in Colab; the zip is beside this notebook.')

---

The figures use three categorical colours validated for colour-vision deficiency at all pairs (worst CVD dE 9.2, normal-vision dE 24.0). Conference papers get printed in greyscale, so every fill also carries a hatch and values are labelled directly, which keeps hue from being the only channel.

All corpora cited in the paper are public. This notebook ships the metric, the parameters and the figure pipeline rather than the data.